code chạy baseline(A)

## Bước 1: Nạp train.json và dev.json, kiểm tra nhãn

Train (8.000 mẫu) dùng để học; dev gốc (3.814 mẫu) dùng để đánh giá. Không tạo dev từ tập con train và không dùng test ở notebook baseline.

Đếm nhãn train để xác nhận dữ liệu trước khi huấn luyện Logistic Regression.


In [1]:
# json đọc từng dòng JSONL và chuyển nó thành dictionary Python.
import json

# Path tạo đường dẫn tệp theo từng thư mục.
from pathlib import Path

# Lấy thư mục hiện tại mà Jupyter đang chạy.
project_root = Path.cwd()

# Nếu notebook chạy từ notebooks/, thư mục cha là gốc dự án.
if project_root.name == "notebooks":
    project_root = project_root.parent

# Tạo đường dẫn tới validation đã chuẩn bị sẵn.
dev_path = project_root / "data" / "processed" / "dev.json"

# Tạo danh sách rỗng để chứa các mẫu validation.
dev_examples = []

# Mở file bằng UTF-8 để đọc đúng tiếng Việt.
with dev_path.open("r", encoding="utf-8") as input_file:
    # Mỗi dòng JSONL là một mẫu QA riêng.
    for line in input_file:
        # Bỏ qua dòng trống để tránh lỗi khi parse JSON.
        if line.strip():
            # json.loads chuyển chuỗi JSON thành dictionary.
            example = json.loads(line)
            dev_examples.append(example)

# Kiểm tra số mẫu đã đọc và các trường của mẫu đầu tiên.
print("Số mẫu validation:", len(dev_examples))
print("Các trường:", dev_examples[0].keys())
print("Câu hỏi:", dev_examples[0]["question"])
print("Đáp án chuẩn:", dev_examples[0]["answers"])
print("Không có đáp án:", dev_examples[0]["is_impossible"])

# Đường dẫn tới tập train dùng để học bộ phân loại.
train_path = project_root / "data" / "processed" / "train.json"
train_examples = []

# Mỗi dòng JSONL là một mẫu QA; đọc lần lượt từng dòng.
with train_path.open("r", encoding="utf-8") as input_file:
    for line in input_file:
        if line.strip():
            train_examples.append(json.loads(line))

# Đếm câu không có đáp án; True được tính là 1.
unanswerable_train_count = sum(
    example["is_impossible"] for example in train_examples
)
answerable_train_count = len(train_examples) - unanswerable_train_count

print("\nSố mẫu train:", len(train_examples))
print("Train — có đáp án:", answerable_train_count)
print("Train — không có đáp án:", unanswerable_train_count)
print("Số mẫu dev:", len(dev_examples))


Số mẫu validation: 3814
Các trường: dict_keys(['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers'])
Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Đáp án chuẩn: {'text': ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí'], 'answer_start': [565]}
Không có đáp án: False

Số mẫu train: 8000
Train — có đáp án: 5417
Train — không có đáp án: 2583
Số mẫu dev: 3814


## BƯỚC 2:
Chia một context thành các câu và tách câu hỏi/câu thành token. Baseline cần các token này để so sánh từ khóa.

In [2]:
# re là thư viện xử lý văn bản theo mẫu, dùng ở đây để tách câu và từ.
import re

# Tách context thành câu bằng dấu ., ! hoặc ? theo sau bởi khoảng trắng.
def split_sentences(context):
    cleaned_context = context.strip()  # Bỏ khoảng trắng thừa ở đầu và cuối.
    sentence_parts = re.split(r"(?<=[.!?])\s+", cleaned_context)

    # Bỏ các phần rỗng để danh sách chỉ chứa câu có nội dung.
    sentences = []
    for sentence in sentence_parts:
        if sentence:
            sentences.append(sentence)

    return sentences

# Đưa văn bản về chữ thường và tách thành các đơn vị chữ/số.
def tokenize_text(text):
    lowercase_text = text.lower()  # Chữ hoa và chữ thường được xem như nhau.
    tokens = re.findall(r"\w+", lowercase_text)  # Dấu câu không được giữ làm token.
    return tokens

# Thử các hàm trên một mẫu validation để xem văn bản được biến đổi ra sao.
example = dev_examples[0]
sentences = split_sentences(example["context"])
question_tokens = tokenize_text(example["question"])

print("Câu hỏi gốc:", example["question"])
print("Token câu hỏi:", question_tokens)
print("\nSố câu trong context:", len(sentences))

# In ba câu đầu để kiểm tra bước tách câu.
for sentence in sentences[:3]:
    print("-", sentence)


""" 
Đây mới là bước chuẩn bị văn bản; ta chưa tính điểm khớp từ khóa. Chạy cell và xem
câu hỏi được tách thành token, còn context thành các câu riêng chưa. Sau đó
ta sẽ viết hàm tính điểm overlap thật đơn giản
"""

Câu hỏi gốc: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Token câu hỏi: ['paris', 'đạt', 'được', 'thành', 'quả', 'gì', 'sau', 'khoảng', '4', 'thế', 'kỷ', 'tính', 'từ', 'ngày', 'cách', 'mạng', 'pháp', 'diễn', 'ra']

Số câu trong context: 5
- Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có.
- Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cùng các cung điện hoàng gia, các tu viện và nhà thờ.
- Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật.


' \nĐây mới là bước chuẩn bị văn bản; ta chưa tính điểm khớp từ khóa. Chạy cell và xem\ncâu hỏi được tách thành token, còn context thành các câu riêng chưa. Sau đó\nta sẽ viết hàm tính điểm overlap thật đơn giản\n'

 ## BƯỚC 3:
Viết thuật toán Jaccard overlap để đo mức độ trùng từ khóa giữa câu hỏi và một câu trong context.
Công thức Jaccard:

$$
J(Q, S) = \frac{|Q \cap S|}{|Q \cup S|}
$$

Trong đó:
- `Q` là tập token của câu hỏi.
- `S` là tập token của câu ứng viên.
- `Q ∩ S` là các token chung.
- `Q ∪ S` là tất cả token riêng biệt của hai tập.

Điểm Jaccard đo mức độ trùng các token giữa câu hỏi và một câu trong context. Trong code của bạn, nó nằm trong khoảng từ 0 đến 1:
- 0: không có token chung.
- Càng gần 1: tỷ lệ token chung càng lớn so với tổng token riêng biệt của hai câu.
Ví dụ, nếu:
    Câu hỏi:  {paris, nổi, tiếng}
    Câu ứng viên: {paris, là, thành, phố}
Hai tập chỉ có token paris chung. Tập hợp tất cả token riêng biệt có 6 phần tử, nên điểm là 1 / 6 ≈ 0.17.
Baseline tính điểm này cho từng câu trong context rồi chọn câu có điểm cao nhất. Điểm Jaccard không phải xác suất câu đó đúng; nó chỉ đo độ giống nhau về từ ngữ. Nếu câu hỏi và câu trả lời diễn đạt cùng ý nhưng dùng từ khác nhau, điểm vẫn có thể thấp.

In [3]:
# Tính Jaccard để đo độ trùng từ khóa giữa câu hỏi và một câu ứng viên.
def calculate_jaccard(question_tokens, candidate_tokens):
    question_set = set(question_tokens)  # set giữ mỗi token một lần.
    candidate_set = set(candidate_tokens)

    common_tokens = question_set.intersection(candidate_set)  # Các token có ở cả hai câu.
    all_tokens = question_set.union(candidate_set)  # Tất cả token của hai câu, không lặp.

    # Nếu cả hai câu đều không có token, tránh phép chia cho 0.
    if len(all_tokens) == 0:
        return 0.0

    # Điểm overlap bằng số token chung chia cho tổng số token riêng biệt.
    score = len(common_tokens) / len(all_tokens)
    return score


# Thử chấm điểm từng câu trong context của mẫu đầu tiên.
best_sentence = ""
best_score = -1.0

for sentence in sentences:
    candidate_tokens = tokenize_text(sentence)
    score = calculate_jaccard(question_tokens, candidate_tokens)

    # Giữ câu có điểm cao nhất làm dự đoán sơ bộ của baseline.
    if score > best_score:
        best_score = score
        best_sentence = sentence

print("Câu hỏi:", example["question"])
print("Điểm Jaccard cao nhất:", best_score)
print("Câu được chọn:", best_sentence)

Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Điểm Jaccard cao nhất: 0.2
Câu được chọn: Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu.


## BƯỚC 4:
Sau khi công thức hiển thị đúng, bước tiếp theo là đóng gói thuật toán vào một hàm dự đoán để có thể dùng lại cho từng câu hỏi.
Ta quy ước ban đầu: nếu không tìm được token chung giữa câu hỏi và bất kỳ câu nào trong context, baseline trả về chuỗi rỗng. Đây là cách xử lý no-answer đơn giản. Ta sẽ đánh giá xem quy tắc này hoạt động ra sao

In [4]:
# Dự đoán câu trả lời bằng câu trong context có Jaccard cao nhất.
def predict_baseline_answer(question, context):
    # Biến câu hỏi thành danh sách token để so khớp.
    question_tokens = tokenize_text(question)

    # Tách context thành các câu ứng viên.
    candidate_sentences = split_sentences(context)

    # Chưa tìm được câu tốt nhất: văn bản rỗng, điểm bằng 0.
    best_sentence = ""
    best_score = 0.0

    # Tính Jaccard giữa câu hỏi và từng câu ứng viên.
    for sentence in candidate_sentences:
        candidate_tokens = tokenize_text(sentence)
        score = calculate_jaccard(question_tokens, candidate_tokens)

        # Chỉ thay câu tốt nhất khi điểm cao hơn điểm đang giữ.
        if score > best_score:
            best_score = score
            best_sentence = sentence

    # Trả cả câu dự đoán và điểm để sau này kiểm tra độ tin cậy.
    return best_sentence, best_score


# Thử hàm trên mẫu đầu tiên trước khi áp dụng cho toàn bộ validation.
example = dev_examples[0]
predicted_answer, prediction_score = predict_baseline_answer(
    example["question"],
    example["context"],
)

print("Câu hỏi:", example["question"])
print("Đáp án chuẩn:", example["answers"]["text"])
print("Nhãn không có đáp án:", example["is_impossible"])
print("Câu baseline chọn:", predicted_answer)
print("Điểm Jaccard:", prediction_score)

Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Đáp án chuẩn: ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí']
Nhãn không có đáp án: False
Câu baseline chọn: Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu.
Điểm Jaccard: 0.2


## BƯỚC 5:
Chạy hàm baseline trên toàn bộ validation (3.814 mẫu) và lưu mỗi dự đoán cùng điểm Jaccard. Đây chỉ là so khớp từ khóa nên chạy trên CPU được, không cần GPU.

In [5]:
# Danh sách này sẽ chứa dự đoán của baseline cho từng mẫu validation.
baseline_predictions = []

# Duyệt qua từng câu hỏi trong validation.
for example in dev_examples:
    # Dùng hàm vừa viết để lấy câu được chọn và điểm Jaccard.
    predicted_answer, prediction_score = predict_baseline_answer(
        example["question"],
        example["context"],
    )

    # Lưu mã mẫu, dự đoán và điểm để đánh giá về sau.
    prediction_row = {
        "id": example["id"],
        "prediction_text": predicted_answer,
        "jaccard_score": prediction_score,
    }
    baseline_predictions.append(prediction_row)

# Kiểm tra baseline đã tạo một dự đoán cho mỗi mẫu chưa.
print("Số mẫu validation:", len(dev_examples))
print("Số dự đoán baseline:", len(baseline_predictions))

# Xem 5 dự đoán đầu tiên để kiểm tra kết quả bằng mắt.
for index in range(5):
    example = dev_examples[index]
    prediction = baseline_predictions[index]

    print("\nCâu hỏi:", example["question"])
    print("Đáp án chuẩn:", example["answers"]["text"])
    print("Baseline chọn:", prediction["prediction_text"])
    print("Điểm Jaccard:", prediction["jaccard_score"])

Số mẫu validation: 3814
Số dự đoán baseline: 3814

Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Đáp án chuẩn: ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí']
Baseline chọn: Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu.
Điểm Jaccard: 0.2

Câu hỏi: Vị trí địa lý của Pháp có gì đặc biệt?
Đáp án chuẩn: []
Baseline chọn: Đến thế kỷ 19 và 20, thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.
Điểm Jaccard: 0.0625

Câu hỏi: Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?
Đáp án chuẩn: ['giáo dục và nghệ thuật', 'nông nghiệp']
Baseline chọn: Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật.
Điểm Jaccard: 0.03333333333333333

Câu hỏi: Cuộc cách mạng Pháp diễn ra ở đ

## Bước 6:
Viết cách đo dự đoán có khớp đáp án chuẩn đến mức nào bằng EM và F1. Vì một câu hỏi có thể có nhiều đáp án chuẩn, ta sẽ so với từng đáp án rồi lấy điểm cao nhất.

In [6]:
# Counter đếm số lần mỗi token xuất hiện, cần thiết để tính F1 khi có từ lặp.
from collections import Counter


# Chuẩn hóa câu trả lời để khác biệt chữ hoa và dấu câu không ảnh hưởng điểm.
def normalize_for_metric(text):
    lowercase_text = text.lower()
    tokens = re.findall(r"\w+", lowercase_text)
    normalized_text = " ".join(tokens)
    return normalized_text


# Exact Match (EM) bằng 1 nếu dự đoán trùng hoàn toàn với đáp án chuẩn.
def calculate_exact_match(predicted_text, gold_text):
    normalized_prediction = normalize_for_metric(predicted_text)
    normalized_gold = normalize_for_metric(gold_text)

    if normalized_prediction == normalized_gold:
        return 1.0

    return 0.0


# Token F1 cân bằng giữa precision và recall của các token trùng nhau.
def calculate_token_f1(predicted_text, gold_text):
    predicted_tokens = normalize_for_metric(predicted_text).split()
    gold_tokens = normalize_for_metric(gold_text).split()

    # Hai chuỗi rỗng nghĩa là dự đoán no-answer đúng.
    if len(predicted_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0

    # Chỉ một chuỗi rỗng nghĩa là một bên trả lời, bên kia để trống.
    if len(predicted_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    # Counter giữ số lần lặp; phép & lấy số token chung theo số lần xuất hiện.
    common_tokens = Counter(predicted_tokens) & Counter(gold_tokens)
    number_of_common_tokens = sum(common_tokens.values())

    if number_of_common_tokens == 0:
        return 0.0

    # Precision: trong dự đoán, bao nhiêu phần token là đúng.
    precision = number_of_common_tokens / len(predicted_tokens)

    # Recall: trong đáp án chuẩn, tìm lại được bao nhiêu phần token.
    recall = number_of_common_tokens / len(gold_tokens)

    # F1 là trung bình điều hòa của precision và recall.
    f1_score = 2 * precision * recall / (precision + recall)
    return f1_score


# Nếu một câu hỏi có nhiều đáp án chuẩn, lấy điểm tốt nhất trong số đó.
def score_prediction(predicted_text, gold_answers):
    if len(gold_answers) == 0:
        gold_answers = [""]  # answers rỗng biểu thị câu hỏi không có đáp án.

    best_em = 0.0
    best_f1 = 0.0

    for gold_text in gold_answers:
        em = calculate_exact_match(predicted_text, gold_text)
        f1 = calculate_token_f1(predicted_text, gold_text)

        if em > best_em:
            best_em = em
        if f1 > best_f1:
            best_f1 = f1

    return best_em, best_f1

""" 
EM chấm nghiêm: câu trả lời phải khớp hoàn toàn sau chuẩn hóa. F1 chấm mức độ token
trùng nhau, nên một câu dự đoán dài hơn đáp án chuẩn vẫn có thể nhận điểm một phần.
"""

' \nEM chấm nghiêm: câu trả lời phải khớp hoàn toàn sau chuẩn hóa. F1 chấm mức độ token\ntrùng nhau, nên một câu dự đoán dài hơn đáp án chuẩn vẫn có thể nhận điểm một phần.\n'

## BƯỚC 7:
Dùng các hàm EM/F1 vừa viết để chấm từng dự đoán trên validation, rồi tính điểm trung bình. Cell này cũng tách riêng câu có đáp án và không có đáp án để thấy baseline đang làm tốt ở nhóm nào.

In [7]:
# Tạo danh sách để gom điểm của tất cả mẫu.
all_em_scores = []
all_f1_scores = []

# Tạo danh sách điểm riêng cho câu có đáp án.
answerable_em_scores = []
answerable_f1_scores = []

# Tạo danh sách điểm riêng cho câu không có đáp án.
unanswerable_em_scores = []
unanswerable_f1_scores = []

# zip ghép mẫu validation với dự đoán tương ứng theo thứ tự.
for example, prediction in zip(dev_examples, baseline_predictions):
    gold_answers = example["answers"]["text"]
    predicted_text = prediction["prediction_text"]

    # Tính EM và F1; nếu có nhiều đáp án chuẩn, hàm lấy điểm tốt nhất.
    em, f1 = score_prediction(predicted_text, gold_answers)

    # Gom điểm chung để tính kết quả trên toàn validation.
    all_em_scores.append(em)
    all_f1_scores.append(f1)

    # Phân nhóm theo nhãn để xem riêng từng loại câu hỏi.
    if example["is_impossible"]:
        unanswerable_em_scores.append(em)
        unanswerable_f1_scores.append(f1)
    else:
        answerable_em_scores.append(em)
        answerable_f1_scores.append(f1)


# Tính trung bình điểm và đổi sang thang phần trăm.
def average_as_percentage(scores):
    if len(scores) == 0:
        return 0.0

    return sum(scores) / len(scores) * 100


# In EM/F1 tổng thể và theo từng nhóm.
print("Số mẫu được chấm:", len(all_em_scores))
print(f"EM tổng thể: {average_as_percentage(all_em_scores):.2f}")
print(f"F1 tổng thể: {average_as_percentage(all_f1_scores):.2f}")

print(f"EM câu có đáp án: {average_as_percentage(answerable_em_scores):.2f}")
print(f"F1 câu có đáp án: {average_as_percentage(answerable_f1_scores):.2f}")

print(f"EM câu không có đáp án: {average_as_percentage(unanswerable_em_scores):.2f}")
print(f"F1 câu không có đáp án: {average_as_percentage(unanswerable_f1_scores):.2f}")

Số mẫu được chấm: 3814
EM tổng thể: 1.13
F1 tổng thể: 22.48
EM câu có đáp án: 1.62
F1 câu có đáp án: 32.32
EM câu không có đáp án: 0.00
F1 câu không có đáp án: 0.00


## BƯỚC 8
Xem các dự đoán sai để hiểu baseline khớp từ khóa thường mắc lỗi gì. Cell này in tối đa 10 mẫu có EM dưới 1:

In [8]:
# Đếm số lỗi đã hiển thị để không in quá nhiều dữ liệu.
shown_errors = 0
maximum_errors_to_show = 10

# Duyệt các mẫu theo đúng thứ tự đã dùng khi tính EM/F1.
for index in range(len(dev_examples)):
    # Bỏ qua mẫu có dự đoán khớp hoàn toàn.
    if all_em_scores[index] == 1.0:
        continue

    example = dev_examples[index]
    prediction = baseline_predictions[index]

    # Hiển thị đáp án rỗng dưới dạng dễ đọc hơn.
    gold_answers = example["answers"]["text"]
    if len(gold_answers) == 0:
        gold_display = "<không có đáp án>"
    else:
        gold_display = gold_answers

    predicted_text = prediction["prediction_text"]
    if predicted_text == "":
        predicted_text = "<không có đáp án>"

    print("\nCâu hỏi:", example["question"])
    print("Loại câu:", "không có đáp án" if example["is_impossible"] else "có đáp án")
    print("Đáp án chuẩn:", gold_display)
    print("Baseline chọn:", predicted_text)
    print("F1:", f"{all_f1_scores[index]:.2f}")
    print("Điểm Jaccard:", f"{prediction['jaccard_score']:.3f}")

    shown_errors += 1

    # Dừng sau 10 lỗi để output vẫn dễ đọc.
    if shown_errors == maximum_errors_to_show:
        break


Câu hỏi: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
Loại câu: có đáp án
Đáp án chuẩn: ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí']
Baseline chọn: Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu.
F1: 0.19
Điểm Jaccard: 0.200

Câu hỏi: Vị trí địa lý của Pháp có gì đặc biệt?
Loại câu: không có đáp án
Đáp án chuẩn: <không có đáp án>
Baseline chọn: Đến thế kỷ 19 và 20, thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.
F1: 0.00
Điểm Jaccard: 0.062

Câu hỏi: Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?
Loại câu: có đáp án
Đáp án chuẩn: ['giáo dục và nghệ thuật', 'nông nghiệp']
Baseline chọn: Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật.
F1: 0.38
Điểm Jaccard: 0.

## Bước 9: Lưu kết quả Jaccard

Metrics, dự đoán và ví dụ lỗi của baseline Jaccard được lưu riêng trong results/.


In [9]:
# csv dùng để lưu bảng dự đoán và mở lại bằng Excel.
import csv

# Tạo thư mục results nếu thư mục này chưa tồn tại.
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Tạo bảng chỉ số chung và theo từng nhóm câu hỏi.
metrics = {
    "all": {
        "count": len(all_em_scores),
        "EM_percent": round(average_as_percentage(all_em_scores), 2),
        "F1_percent": round(average_as_percentage(all_f1_scores), 2),
    },
    "answerable": {
        "count": len(answerable_em_scores),
        "EM_percent": round(average_as_percentage(answerable_em_scores), 2),
        "F1_percent": round(average_as_percentage(answerable_f1_scores), 2),
    },
    "unanswerable": {
        "count": len(unanswerable_em_scores),
        "EM_percent": round(average_as_percentage(unanswerable_em_scores), 2),
        "F1_percent": round(average_as_percentage(unanswerable_f1_scores), 2),
    },
}

# Lưu EM/F1 thành JSON để có thể đọc lại bằng Python hoặc trình soạn thảo.
metrics_path = results_dir / "baseline_metrics.json"
with metrics_path.open("w", encoding="utf-8") as output_file:
    json.dump(metrics, output_file, ensure_ascii=False, indent=2)

# Gom thông tin dự đoán của từng mẫu để lưu thành CSV.
prediction_rows = []
for index, (example, prediction) in enumerate(zip(dev_examples, baseline_predictions)):
    prediction_rows.append({
        "id": example["id"],
        "is_impossible": example["is_impossible"],
        "question": example["question"],
        "gold_answers": json.dumps(example["answers"]["text"], ensure_ascii=False),
        "prediction_text": prediction["prediction_text"],
        "jaccard_score": prediction["jaccard_score"],
        "em": all_em_scores[index],
        "f1": all_f1_scores[index],
    })

# Ghi bảng dự đoán đầy đủ; utf-8-sig giúp Excel mở tiếng Việt thuận tiện hơn.
predictions_path = results_dir / "baseline_predictions.csv"
with predictions_path.open("w", encoding="utf-8-sig", newline="") as output_file:
    column_names = prediction_rows[0].keys()
    writer = csv.DictWriter(output_file, fieldnames=column_names)
    writer.writeheader()
    writer.writerows(prediction_rows)

# Lấy tối đa 5 lỗi mỗi nhóm để lưu riêng cho việc xem lại.
error_rows = []
for group_name, is_impossible_value in [
    ("answerable_error", False),
    ("unanswerable_error", True),
]:
    group_error_count = 0
    for index, (example, prediction) in enumerate(zip(dev_examples, baseline_predictions)):
        if example["is_impossible"] == is_impossible_value and all_em_scores[index] == 0.0:
            error_rows.append({
                "error_group": group_name,
                "id": example["id"],
                "question": example["question"],
                "gold_answers": json.dumps(example["answers"]["text"], ensure_ascii=False),
                "prediction_text": prediction["prediction_text"],
                "jaccard_score": prediction["jaccard_score"],
                "f1": all_f1_scores[index],
            })
            group_error_count += 1
            if group_error_count == 5:
                break

# Lưu ví dụ lỗi để xem lại sau khi chạy notebook.
errors_path = results_dir / "baseline_error_examples.csv"
error_columns = ["error_group", "id", "question", "gold_answers", "prediction_text", "jaccard_score", "f1"]
with errors_path.open("w", encoding="utf-8-sig", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=error_columns)
    writer.writeheader()
    writer.writerows(error_rows)

# In kết quả và đường dẫn tệp để xác nhận đã lưu.
for group_name, group_metrics in metrics.items():
    print(
        group_name,
        "| số mẫu:", group_metrics["count"],
        "| EM:", group_metrics["EM_percent"], "%",
        "| F1:", group_metrics["F1_percent"], "%",
    )
print("Đã lưu chỉ số:", metrics_path)
print("Đã lưu dự đoán:", predictions_path)
print("Đã lưu ví dụ lỗi:", errors_path)


all | số mẫu: 3814 | EM: 1.13 % | F1: 22.48 %
answerable | số mẫu: 2653 | EM: 1.62 % | F1: 32.32 %
unanswerable | số mẫu: 1161 | EM: 0.0 % | F1: 0.0 %
Đã lưu chỉ số: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_metrics.json
Đã lưu dự đoán: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_predictions.csv
Đã lưu ví dụ lỗi: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_error_examples.csv


## Bước 10: Tạo đặc trưng TF-IDF và huấn luyện Logistic Regression

TF-IDF biến văn bản thành vector số; từ quan trọng trong một mẫu nhưng ít phổ biến toàn tập thường có trọng số cao hơn. Logistic Regression học từ các vector train để dự đoán is_impossible.

Bộ phân loại quyết định có/không có đáp án. Nếu có đáp án, bước sau dùng TF-IDF chọn câu trong context. Chỉ fit vectorizer trên train để dev không tham gia học.


In [10]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# TfidfVectorizer chuyển văn bản thành vector đặc trưng thưa.
from sklearn.feature_extraction.text import TfidfVectorizer
# LogisticRegression là bộ phân loại nhị phân truyền thống.
from sklearn.linear_model import LogisticRegression

# Nối câu hỏi và context để biểu diễn toàn bộ thông tin của mẫu.
train_texts = []
train_labels = []
for example in train_examples:
    combined_text = example["question"] + " " + example["context"]
    train_texts.append(combined_text)
    train_labels.append(bool(example["is_impossible"]))

# Học từ điển và trọng số TF-IDF chỉ từ văn bản train.
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
train_tfidf = tfidf_vectorizer.fit_transform(train_texts)

# Cân bằng ảnh hưởng hai nhãn; random_state giúp kết quả lặp lại được.
answerability_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
)
answerability_model.fit(train_tfidf, train_labels)

print("Số mẫu học:", train_tfidf.shape[0])
print("Số đặc trưng TF-IDF:", train_tfidf.shape[1])
print("Các nhãn đã học:", answerability_model.classes_)


Số mẫu học: 8000
Số đặc trưng TF-IDF: 50000
Các nhãn đã học: [False  True]


## Bước 11: Dự đoán no-answer và chọn câu bằng TF-IDF

Logistic Regression dự đoán trước câu hỏi có đáp án hay không. Nếu không có, trả chuỗi rỗng. Nếu có, cosine similarity so vector của câu hỏi với từng câu trong context rồi chọn câu gần nhất.


In [12]:
# Cosine similarity so sánh hướng của hai vector TF-IDF.
from sklearn.metrics.pairwise import cosine_similarity

def predict_tfidf_logreg(question, context):
    # Dự đoán nhãn có/không có đáp án từ câu hỏi và context.
    combined_text = question + " " + context
    combined_vector = tfidf_vectorizer.transform([combined_text])
    predicted_unanswerable = bool(answerability_model.predict(combined_vector)[0])

    # Nếu dự đoán không có đáp án, kết quả là chuỗi rỗng.
    if predicted_unanswerable:
        return "", 0.0, predicted_unanswerable

    # Nếu có đáp án, chia context thành các câu ứng viên.
    candidate_sentences = split_sentences(context)
    if len(candidate_sentences) == 0:
        return "", 0.0, predicted_unanswerable

    # Biến câu hỏi và từng câu ứng viên thành vector cùng từ điển TF-IDF.
    question_vector = tfidf_vectorizer.transform([question])
    sentence_vectors = tfidf_vectorizer.transform(candidate_sentences)
    similarity_scores = cosine_similarity(question_vector, sentence_vectors)[0]

    # Lấy câu có điểm tương đồng cao nhất làm câu trả lời sơ bộ.
    best_index = int(similarity_scores.argmax())
    best_sentence = candidate_sentences[best_index]
    best_score = float(similarity_scores[best_index])
    return best_sentence, best_score, predicted_unanswerable

# Xem một ví dụ trước khi chạy trên toàn bộ dev.
example = dev_examples[0]
predicted_text, similarity, predicted_no_answer = predict_tfidf_logreg(
    example["question"], example["context"]
)
print("Dự đoán:", predicted_text)
print("Dự đoán không có đáp án:", predicted_no_answer)
print("Cosine similarity:", similarity)


Dự đoán: Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu.
Dự đoán không có đáp án: False
Cosine similarity: 0.19220461526185034


## Bước 12: Chạy baseline mới trên toàn bộ dev

Chạy trên cùng 3.814 mẫu dev như Jaccard để so sánh công bằng.


In [13]:
# Tạo một dự đoán cho mỗi mẫu dev, giữ nguyên thứ tự.
tfidf_predictions = []
for example in dev_examples:
    predicted_text, similarity, predicted_no_answer = predict_tfidf_logreg(
        example["question"], example["context"]
    )
    prediction_row = {
        "id": example["id"],
        "prediction_text": predicted_text,
        "predicted_is_impossible": predicted_no_answer,
        "cosine_similarity": similarity,
    }
    tfidf_predictions.append(prediction_row)

print("Số mẫu dev:", len(dev_examples))
print("Số dự đoán:", len(tfidf_predictions))


Số mẫu dev: 3814
Số dự đoán: 3814


## Bước 13: Tính EM, F1 và đánh giá phân loại có/không có đáp án

EM/F1 chấm đoạn văn được trả về. Accuracy phân loại là tỷ lệ mẫu mà bộ phân loại đoán đúng nhãn có/không có đáp án trên toàn dev. Nó không riêng cho nhóm không có đáp án.

Để biết mô hình tìm được bao nhiêu câu thật sự không có đáp án, xem thêm recall của nhãn không có đáp án. Confusion matrix có thứ tự hàng/cột là False (có đáp án), True (không có đáp án).


In [14]:
# Dùng scikit-learn để tính accuracy và confusion matrix.
from sklearn.metrics import accuracy_score, confusion_matrix

tfidf_em_scores = []
tfidf_f1_scores = []
true_no_answer = []
predicted_no_answer = []

# So dự đoán với đáp án chuẩn và lưu nhãn no-answer hai phía.
for example, prediction in zip(dev_examples, tfidf_predictions):
    em, f1 = score_prediction(
        prediction["prediction_text"], example["answers"]["text"]
    )
    tfidf_em_scores.append(em)
    tfidf_f1_scores.append(f1)
    true_no_answer.append(bool(example["is_impossible"]))
    predicted_no_answer.append(prediction["predicted_is_impossible"])

# Cố định thứ tự lớp: answerable=False, unanswerable=True.
no_answer_matrix = confusion_matrix(
    true_no_answer, predicted_no_answer, labels=[False, True]
)

# Accuracy là tỷ lệ dự đoán đúng trên cả hai nhóm câu hỏi.
answerability_accuracy = accuracy_score(true_no_answer, predicted_no_answer)

# Recall của nhãn True cho biết phát hiện được bao nhiêu câu no-answer thật.
unanswerable_total = int(no_answer_matrix[1].sum())
if unanswerable_total > 0:
    unanswerable_recall = no_answer_matrix[1, 1] / unanswerable_total
else:
    unanswerable_recall = 0.0

print(f"EM dev: {average_as_percentage(tfidf_em_scores):.2f}")
print(f"F1 dev: {average_as_percentage(tfidf_f1_scores):.2f}")
print(f"Accuracy phân loại có/không có đáp án: {answerability_accuracy * 100:.2f}%")
print(f"Recall nhãn không có đáp án: {unanswerable_recall * 100:.2f}%")
print("Confusion matrix [thật: False/True, dự đoán: False/True]:")
print(no_answer_matrix)


EM dev: 7.87
F1 dev: 23.14
Accuracy phân loại có/không có đáp án: 58.36%
Recall nhãn không có đáp án: 23.43%
Confusion matrix [thật: False/True, dự đoán: False/True]:
[[1954  699]
 [ 889  272]]


## Bước 14: Lưu kết quả TF-IDF + Logistic Regression

Lưu kết quả vào các file riêng để không ghi đè Jaccard. Đây là kết quả trên dev; không dùng test.json.


In [15]:
# Tạo thư mục results nếu chưa có.
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Gom các metric và thông số quan trọng để lưu thành JSON.
tfidf_metrics = {
    "model": "TF-IDF + LogisticRegression + TF-IDF sentence ranking",
    "split": "dev",
    "count": len(dev_examples),
    "EM_percent": round(average_as_percentage(tfidf_em_scores), 2),
    "F1_percent": round(average_as_percentage(tfidf_f1_scores), 2),
    "answerability_accuracy_percent": round(answerability_accuracy * 100, 2),
    "unanswerable_recall_percent": round(unanswerable_recall * 100, 2),
    "confusion_matrix_labels": ["answerable_False", "unanswerable_True"],
    "confusion_matrix": no_answer_matrix.tolist(),
    "random_state": 42,
    "max_features": 50000,
    "ngram_range": [1, 2],
}
metrics_path = results_dir / "baseline_tfidf_logreg_metrics.json"
with metrics_path.open("w", encoding="utf-8") as output_file:
    json.dump(tfidf_metrics, output_file, ensure_ascii=False, indent=2)

# Tạo từng dòng dự đoán tường minh để dễ đọc và phân tích lỗi.
prediction_rows = []
for example, prediction in zip(dev_examples, tfidf_predictions):
    prediction_row = {
        "id": example["id"],
        "question": example["question"],
        "is_impossible": bool(example["is_impossible"]),
        "prediction_text": prediction["prediction_text"],
        "predicted_is_impossible": prediction["predicted_is_impossible"],
        "cosine_similarity": prediction["cosine_similarity"],
    }
    prediction_rows.append(prediction_row)

predictions_path = results_dir / "baseline_tfidf_logreg_predictions.json"
with predictions_path.open("w", encoding="utf-8") as output_file:
    json.dump(prediction_rows, output_file, ensure_ascii=False, indent=2)

print("Đã lưu metrics:", metrics_path)
print("Đã lưu dự đoán:", predictions_path)


Đã lưu metrics: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_tfidf_logreg_metrics.json
Đã lưu dự đoán: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_tfidf_logreg_predictions.json


## Bước 15: So sánh hai baseline trên dev

Cell kế tiếp tạo bảng so sánh Jaccard với TF-IDF + Logistic Regression trên cùng tập dev. Bảng gồm EM/F1, accuracy phân loại answerability và recall riêng cho nhãn không có đáp án. Nó cũng lưu confusion matrix và các metric so sánh vào results/baseline_comparison_metrics.json.

Chỉ dùng dev để so sánh phương pháp. Giữ test.json cho đánh giá cuối cùng sau khi chọn baseline. Hai phương pháp hiện chọn nguyên câu context làm dự đoán, không trích chính xác một cụm con như mô hình extractive QA; hãy nêu giới hạn này khi diễn giải EM/F1.


In [16]:
# Tạo nhãn thật và nhãn dự đoán answerability cho Jaccard.
# Jaccard trả chuỗi rỗng khi không tìm thấy token chung; đó là dự đoán no-answer.
jaccard_true_no_answer = []
jaccard_predicted_no_answer = []

for example, prediction in zip(dev_examples, baseline_predictions):
    jaccard_true_no_answer.append(bool(example["is_impossible"]))
    jaccard_predicted_no_answer.append(
        prediction["prediction_text"] == ""
    )

# Hàng là nhãn thật, cột là nhãn dự đoán; thứ tự lớp là False rồi True.
jaccard_matrix = confusion_matrix(
    jaccard_true_no_answer,
    jaccard_predicted_no_answer,
    labels=[False, True],
)
jaccard_answerability_accuracy = accuracy_score(
    jaccard_true_no_answer,
    jaccard_predicted_no_answer,
)

# Recall = số câu no-answer nhận ra đúng / tổng số câu no-answer thật.
jaccard_unanswerable_total = int(jaccard_matrix[1].sum())
if jaccard_unanswerable_total > 0:
    jaccard_unanswerable_recall = (
        jaccard_matrix[1, 1] / jaccard_unanswerable_total
    )
else:
    jaccard_unanswerable_recall = 0.0

# Gom metric của hai phương pháp để in bảng và lưu kết quả.
comparison = {
    "split": "dev",
    "count": len(dev_examples),
    "Jaccard": {
        "EM_percent": metrics["all"]["EM_percent"],
        "F1_percent": metrics["all"]["F1_percent"],
        "answerability_accuracy_percent": round(
            jaccard_answerability_accuracy * 100, 2
        ),
        "unanswerable_recall_percent": round(
            jaccard_unanswerable_recall * 100, 2
        ),
        "confusion_matrix": jaccard_matrix.tolist(),
    },
    "TF-IDF + Logistic Regression": {
        "EM_percent": tfidf_metrics["EM_percent"],
        "F1_percent": tfidf_metrics["F1_percent"],
        "answerability_accuracy_percent": tfidf_metrics[
            "answerability_accuracy_percent"
        ],
        "unanswerable_recall_percent": tfidf_metrics[
            "unanswerable_recall_percent"
        ],
        "confusion_matrix": tfidf_metrics["confusion_matrix"],
    },
}

# In bảng gọn để dễ đối chiếu ngay trong notebook.
print("So sánh trên dev:", comparison["count"], "mẫu")
print(
    f"{'Phương pháp':34}"
    f"{'EM':>10}"
    f"{'F1':>10}"
    f"{'Accuracy':>14}"
    f"{'Recall no-answer':>20}"
)
for method_name in ["Jaccard", "TF-IDF + Logistic Regression"]:
    method_metrics = comparison[method_name]
    print(
        f"{method_name:34}"
        f"{method_metrics['EM_percent']:>9.2f}%"
        f"{method_metrics['F1_percent']:>9.2f}%"
        f"{method_metrics['answerability_accuracy_percent']:>13.2f}%"
        f"{method_metrics['unanswerable_recall_percent']:>19.2f}%"
    )
    print("  Confusion matrix [thật: False/True, dự đoán: False/True]:")
    print(method_metrics["confusion_matrix"])

# Lưu bảng số liệu để dùng trong báo cáo mà không cần chép từ output.
comparison_path = results_dir / "baseline_comparison_metrics.json"
with comparison_path.open("w", encoding="utf-8") as output_file:
    json.dump(comparison, output_file, ensure_ascii=False, indent=2)

print("Đã lưu bảng so sánh:", comparison_path)


So sánh trên dev: 3814 mẫu
Phương pháp                               EM        F1      Accuracy    Recall no-answer
Jaccard                                1.13%    22.48%        69.56%               0.00%
  Confusion matrix [thật: False/True, dự đoán: False/True]:
[[2653, 0], [1161, 0]]
TF-IDF + Logistic Regression           7.87%    23.14%        58.36%              23.43%
  Confusion matrix [thật: False/True, dự đoán: False/True]:
[[1954, 699], [889, 272]]
Đã lưu bảng so sánh: c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\baseline_comparison_metrics.json


## Kết quả baseline trên dev

Cả hai baseline đã chạy trên cùng **3.814 mẫu dev**. Tập test chưa được dùng.

| Phương pháp | EM | F1 | Accuracy phân loại có/không có đáp án | Recall nhãn không có đáp án |
|---|---:|---:|---:|---:|
| Jaccard | 1,13% | 22,48% | 69,56% | 0,00% |
| TF-IDF + Logistic Regression | 7,87% | 23,14% | 58,36% | 23,43% |

Jaccard chọn nguyên câu context có độ trùng từ cao nhất. Trên dev, phương pháp này dự đoán **tất cả mẫu đều có đáp án**, nên accuracy phân loại đạt 69,56% do nhóm có đáp án chiếm đa số, nhưng recall cho nhóm không có đáp án là 0%.

TF-IDF + Logistic Regression đạt EM và F1 cao hơn Jaccard, đồng thời nhận ra được một phần câu không có đáp án. Tuy vậy, accuracy phân loại thấp hơn và recall no-answer mới đạt 23,43%; baseline này vẫn còn nhiều câu không có đáp án bị phân loại nhầm.

Cả hai phương pháp chọn nguyên câu trong context thay vì trích chính xác một cụm đáp án. Vì vậy, EM thấp và F1 chỉ phản ánh mức độ token trùng một phần. Đây là các baseline để so sánh với Zero-shot và fine-tuning; chưa phải kết quả cuối trên test.

Kết quả so sánh đã lưu tại results/baseline_comparison_metrics.json.
